In [1]:
# ============================================================
# 1. Install and import libraries
# ============================================================

!pip install shapely -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.colab import files
from shapely import wkt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# ============================================================
# 2. Upload dataset
# ============================================================

print("Please upload your CSV dataset file.")
uploaded = files.upload()

file_name = list(uploaded.keys())[0]
print("Uploaded file:", file_name)

df = pd.read_csv(file_name)

print("Original dataset shape:", df.shape)
display(df.head())

In [ ]:
# ============================================================
# 3. Define target variable and remove missing target rows
# ============================================================

target = "PM25_ugm3"

if target not in df.columns:
    raise ValueError(f"Target variable '{target}' not found in the dataset.")

df = df.dropna(subset=[target]).copy()

print("Dataset after removing missing PM2.5 target:", df.shape)


In [ ]:
# ============================================================
# 4. Extract spatial features from road geometry
# ============================================================

if "the_geom" not in df.columns:
    raise ValueError("Column 'the_geom' not found. Geometry is needed to extract latitude and longitude.")

df["geometry"] = df["the_geom"].apply(wkt.loads)

df["midpoint"] = df["geometry"].apply(
    lambda x: x.interpolate(0.5, normalized=True)
)

df["longitude"] = df["midpoint"].apply(lambda p: p.x)
df["latitude"] = df["midpoint"].apply(lambda p: p.y)

print("Extracted spatial features:")
display(df[["longitude", "latitude"]].head())

In [ ]:
# ============================================================
# 5. Create temporal features if any date/time column exists
# ============================================================

possible_time_cols = [
    col for col in df.columns
    if "date" in col.lower() or "time" in col.lower() or "hour" in col.lower()
]

print("Detected possible temporal columns:", possible_time_cols)

if len(possible_time_cols) > 0:
    time_col = possible_time_cols[0]
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")

    df["hour"] = df[time_col].dt.hour
    df["day"] = df[time_col].dt.day
    df["month"] = df[time_col].dt.month
    df["dayofweek"] = df[time_col].dt.dayofweek

    print("Temporal features created from:", time_col)
else:
    print("No timestamp column found. Spatial, road and pollutant/context features will be used.")

In [ ]:
# ============================================================
# 6. Define feature matrix X and target y
# ============================================================

drop_cols = [
    target,
    "road_id",
    "the_geom",
    "geometry",
    "midpoint"
]

drop_cols = [col for col in drop_cols if col in df.columns]

X = df.drop(columns=drop_cols)
y = df[target]

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

In [ ]:
# ============================================================
# 7. Identify numerical and categorical features
# ============================================================

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("Numerical features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

In [ ]:
# ============================================================
# 8. Preprocessing pipeline
# ============================================================

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

# Compatible with newer and older sklearn versions
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", encoder)
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)


In [ ]:
# ============================================================
# 9. Train-test split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)

In [ ]:
# ============================================================
# RO1: Develop Random Forest and Gradient Boosting models
# ============================================================

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    ))
])

gb_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", GradientBoostingRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")
print("Gradient Boosting model trained successfully.")

rf_pred = rf_model.predict(X_test)
gb_pred = gb_model.predict(X_test)

In [ ]:
# ============================================================
# RO2: Evaluate and compare model performance
# ============================================================

def evaluate_model(y_true, y_pred, model_name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return {
        "Model": model_name,
        "RMSE": round(rmse, 4),
        "MAE": round(mae, 4),
        "R²": round(r2, 4)
    }

evaluation_table = pd.DataFrame([
    evaluate_model(y_test, rf_pred, "Random Forest Regressor"),
    evaluate_model(y_test, gb_pred, "Gradient Boosting Regressor")
])

print("Model Evaluation Table:")
display(evaluation_table)

evaluation_table.to_csv("model_evaluation_results_PM25.csv", index=False)

In [ ]:
# ============================================================
# 10. Prediction results table
# ============================================================

prediction_results = pd.DataFrame({
    "Observed_PM25": y_test.values,
    "Random_Forest_Predicted_PM25": rf_pred,
    "Gradient_Boosting_Predicted_PM25": gb_pred
})

prediction_results["RF_Residual"] = (
    prediction_results["Observed_PM25"] -
    prediction_results["Random_Forest_Predicted_PM25"]
)

prediction_results["GB_Residual"] = (
    prediction_results["Observed_PM25"] -
    prediction_results["Gradient_Boosting_Predicted_PM25"]
)

print("Observed vs Predicted Results:")
display(prediction_results.head())

prediction_results.to_csv("observed_vs_predicted_PM25.csv", index=False)

In [ ]:
# ============================================================
# 11. Random Forest evaluation collage
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(
    prediction_results["Observed_PM25"],
    prediction_results["Random_Forest_Predicted_PM25"],
    alpha=0.6
)

min_val = min(
    prediction_results["Observed_PM25"].min(),
    prediction_results["Random_Forest_Predicted_PM25"].min()
)

max_val = max(
    prediction_results["Observed_PM25"].max(),
    prediction_results["Random_Forest_Predicted_PM25"].max()
)

axes[0, 0].plot([min_val, max_val], [min_val, max_val])
axes[0, 0].set_title("Random Forest: Observed vs Predicted PM2.5")
axes[0, 0].set_xlabel("Observed PM2.5")
axes[0, 0].set_ylabel("Predicted PM2.5")

axes[0, 1].scatter(
    prediction_results["Random_Forest_Predicted_PM25"],
    prediction_results["RF_Residual"],
    alpha=0.6
)

axes[0, 1].axhline(0, linestyle="--")
axes[0, 1].set_title("Random Forest: Residual Plot")
axes[0, 1].set_xlabel("Predicted PM2.5")
axes[0, 1].set_ylabel("Residuals")

axes[1, 0].hist(
    prediction_results["RF_Residual"],
    bins=40
)

axes[1, 0].set_title("Random Forest: Residual Distribution")
axes[1, 0].set_xlabel("Residuals")
axes[1, 0].set_ylabel("Frequency")

sample_n = min(250, len(prediction_results))

axes[1, 1].plot(
    prediction_results["Observed_PM25"].values[:sample_n],
    label="Observed PM2.5"
)

axes[1, 1].plot(
    prediction_results["Random_Forest_Predicted_PM25"].values[:sample_n],
    label="Predicted PM2.5"
)

axes[1, 1].set_title("Random Forest: Observed vs Predicted Trend")
axes[1, 1].set_xlabel("Sample Index")
axes[1, 1].set_ylabel("PM2.5")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("random_forest_PM25_evaluation_collage.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 12. Gradient Boosting evaluation collage
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].scatter(
    prediction_results["Observed_PM25"],
    prediction_results["Gradient_Boosting_Predicted_PM25"],
    alpha=0.6
)

min_val = min(
    prediction_results["Observed_PM25"].min(),
    prediction_results["Gradient_Boosting_Predicted_PM25"].min()
)

max_val = max(
    prediction_results["Observed_PM25"].max(),
    prediction_results["Gradient_Boosting_Predicted_PM25"].max()
)

axes[0, 0].plot([min_val, max_val], [min_val, max_val])
axes[0, 0].set_title("Gradient Boosting: Observed vs Predicted PM2.5")
axes[0, 0].set_xlabel("Observed PM2.5")
axes[0, 0].set_ylabel("Predicted PM2.5")

axes[0, 1].scatter(
    prediction_results["Gradient_Boosting_Predicted_PM25"],
    prediction_results["GB_Residual"],
    alpha=0.6
)

axes[0, 1].axhline(0, linestyle="--")
axes[0, 1].set_title("Gradient Boosting: Residual Plot")
axes[0, 1].set_xlabel("Predicted PM2.5")
axes[0, 1].set_ylabel("Residuals")

axes[1, 0].hist(
    prediction_results["GB_Residual"],
    bins=40
)

axes[1, 0].set_title("Gradient Boosting: Residual Distribution")
axes[1, 0].set_xlabel("Residuals")
axes[1, 0].set_ylabel("Frequency")

axes[1, 1].plot(
    prediction_results["Observed_PM25"].values[:sample_n],
    label="Observed PM2.5"
)

axes[1, 1].plot(
    prediction_results["Gradient_Boosting_Predicted_PM25"].values[:sample_n],
    label="Predicted PM2.5"
)

axes[1, 1].set_title("Gradient Boosting: Observed vs Predicted Trend")
axes[1, 1].set_xlabel("Sample Index")
axes[1, 1].set_ylabel("PM2.5")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("gradient_boosting_PM25_evaluation_collage.png", dpi=300)
plt.show()


In [ ]:
# ============================================================
# 13. Combined comparison collage
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].bar(
    evaluation_table["Model"],
    evaluation_table["RMSE"]
)
axes[0, 0].set_title("RMSE Comparison")
axes[0, 0].set_ylabel("RMSE")
axes[0, 0].tick_params(axis="x", rotation=15)

axes[0, 1].bar(
    evaluation_table["Model"],
    evaluation_table["MAE"]
)
axes[0, 1].set_title("MAE Comparison")
axes[0, 1].set_ylabel("MAE")
axes[0, 1].tick_params(axis="x", rotation=15)

axes[1, 0].bar(
    evaluation_table["Model"],
    evaluation_table["R²"]
)
axes[1, 0].set_title("R² Comparison")
axes[1, 0].set_ylabel("R²")
axes[1, 0].tick_params(axis="x", rotation=15)

axes[1, 1].scatter(
    prediction_results["Observed_PM25"],
    prediction_results["Random_Forest_Predicted_PM25"],
    alpha=0.5,
    label="Random Forest"
)

axes[1, 1].scatter(
    prediction_results["Observed_PM25"],
    prediction_results["Gradient_Boosting_Predicted_PM25"],
    alpha=0.5,
    label="Gradient Boosting"
)

min_val = prediction_results["Observed_PM25"].min()
max_val = prediction_results["Observed_PM25"].max()

axes[1, 1].plot([min_val, max_val], [min_val, max_val])
axes[1, 1].set_title("Observed vs Predicted Comparison")
axes[1, 1].set_xlabel("Observed PM2.5")
axes[1, 1].set_ylabel("Predicted PM2.5")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("combined_model_comparison_PM25_collage.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# 14. Best model summary
# ============================================================

best_r2_model = evaluation_table.loc[evaluation_table["R²"].idxmax()]
best_rmse_model = evaluation_table.loc[evaluation_table["RMSE"].idxmin()]

print("Best model based on highest R²:")
display(best_r2_model)

print("Best model based on lowest RMSE:")
display(best_rmse_model)

In [ ]:
# ============================================================
# 15. Download outputs
# ============================================================

print("Downloading saved files...")

files.download("model_evaluation_results_PM25.csv")
files.download("observed_vs_predicted_PM25.csv")
files.download("random_forest_PM25_evaluation_collage.png")
files.download("gradient_boosting_PM25_evaluation_collage.png")
files.download("combined_model_comparison_PM25_collage.png")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# RO3: Feature Importance Analysis
# ============================================================

# ------------------------------------------------------------
# 1. Function to extract feature names after preprocessing
# ------------------------------------------------------------

def get_feature_names_from_pipeline(model_pipeline):
    """
    Extract feature names from preprocessing pipeline.
    Works with numerical and one-hot encoded categorical variables.
    """
    preprocessor = model_pipeline.named_steps["preprocessor"]

    feature_names = []

    # Numerical feature names
    if "num" in preprocessor.named_transformers_:
        num_features = preprocessor.transformers_[0][2]
        feature_names.extend(num_features)

    # Categorical encoded feature names
    if "cat" in preprocessor.named_transformers_:
        cat_features = preprocessor.transformers_[1][2]
        cat_pipeline = preprocessor.named_transformers_["cat"]
        encoder = cat_pipeline.named_steps["onehot"]

        try:
            cat_encoded_features = encoder.get_feature_names_out(cat_features)
        except:
            cat_encoded_features = encoder.get_feature_names(cat_features)

        feature_names.extend(cat_encoded_features)

    return np.array(feature_names)

In [ ]:
# ------------------------------------------------------------
# 2. Extract feature importance from Random Forest
# ------------------------------------------------------------

rf_feature_names = get_feature_names_from_pipeline(rf_model)

rf_importance = rf_model.named_steps["model"].feature_importances_

rf_importance_df = pd.DataFrame({
    "Feature": rf_feature_names,
    "Importance": rf_importance
}).sort_values(by="Importance", ascending=False)

print("Top 20 Random Forest Feature Importance:")
display(rf_importance_df.head(20))

In [ ]:
# ------------------------------------------------------------
# 3. Extract feature importance from Gradient Boosting
# ------------------------------------------------------------

gb_feature_names = get_feature_names_from_pipeline(gb_model)

gb_importance = gb_model.named_steps["model"].feature_importances_

gb_importance_df = pd.DataFrame({
    "Feature": gb_feature_names,
    "Importance": gb_importance
}).sort_values(by="Importance", ascending=False)

print("Top 20 Gradient Boosting Feature Importance:")
display(gb_importance_df.head(20))

In [ ]:
# ------------------------------------------------------------
# 4. Save feature importance tables
# ------------------------------------------------------------

rf_importance_df.to_csv("random_forest_feature_importance_PM25.csv", index=False)
gb_importance_df.to_csv("gradient_boosting_feature_importance_PM25.csv", index=False)

print("Feature importance tables saved.")

In [ ]:
# ------------------------------------------------------------
# 5. Feature importance collage chart
# ------------------------------------------------------------

top_n = 15

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Random Forest feature importance
rf_top = rf_importance_df.head(top_n).sort_values(by="Importance", ascending=True)

axes[0].barh(rf_top["Feature"], rf_top["Importance"])
axes[0].set_title("Top 15 Feature Importance: Random Forest")
axes[0].set_xlabel("Importance Score")
axes[0].set_ylabel("Feature")

# Gradient Boosting feature importance
gb_top = gb_importance_df.head(top_n).sort_values(by="Importance", ascending=True)

axes[1].barh(gb_top["Feature"], gb_top["Importance"])
axes[1].set_title("Top 15 Feature Importance: Gradient Boosting")
axes[1].set_xlabel("Importance Score")
axes[1].set_ylabel("Feature")

plt.tight_layout()
plt.savefig("feature_importance_collage_PM25.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# RO4: Spatial Visualisation of Observed and Predicted PM2.5
# ============================================================

# ------------------------------------------------------------
# 6. Create spatial results dataframe
# ------------------------------------------------------------

spatial_results = X_test.copy()

spatial_results["Observed_PM25"] = y_test.values
spatial_results["RF_Predicted_PM25"] = rf_pred
spatial_results["GB_Predicted_PM25"] = gb_pred

# Residuals
spatial_results["RF_Residual"] = spatial_results["Observed_PM25"] - spatial_results["RF_Predicted_PM25"]
spatial_results["GB_Residual"] = spatial_results["Observed_PM25"] - spatial_results["GB_Predicted_PM25"]

# Check required spatial columns
if "longitude" not in spatial_results.columns or "latitude" not in spatial_results.columns:
    raise ValueError("longitude and latitude columns are required for spatial mapping.")

print("Spatial results preview:")
display(spatial_results[[
    "longitude",
    "latitude",
    "Observed_PM25",
    "RF_Predicted_PM25",
    "GB_Predicted_PM25",
    "RF_Residual",
    "GB_Residual"
]].head())

In [ ]:
# ------------------------------------------------------------
# 7. Save spatial results table
# ------------------------------------------------------------

spatial_results.to_csv("spatial_observed_predicted_PM25.csv", index=False)

print("Spatial observed and predicted PM2.5 table saved.")


# ------------------------------------------------------------
# 8. Define hotspot threshold
# ------------------------------------------------------------

# Persistent hotspots are defined here as locations in the top 10%
# of observed PM2.5 concentration values.

hotspot_threshold = spatial_results["Observed_PM25"].quantile(0.90)

spatial_results["Observed_Hotspot"] = spatial_results["Observed_PM25"] >= hotspot_threshold
spatial_results["RF_Predicted_Hotspot"] = spatial_results["RF_Predicted_PM25"] >= hotspot_threshold
spatial_results["GB_Predicted_Hotspot"] = spatial_results["GB_Predicted_PM25"] >= hotspot_threshold

print("Hotspot threshold based on top 10% observed PM2.5:")
print(round(hotspot_threshold, 3))

print("Observed hotspot count:", spatial_results["Observed_Hotspot"].sum())
print("RF predicted hotspot count:", spatial_results["RF_Predicted_Hotspot"].sum())
print("GB predicted hotspot count:", spatial_results["GB_Predicted_Hotspot"].sum())

In [ ]:
# ------------------------------------------------------------
# 9. Spatial collage: observed and predicted PM2.5
# ------------------------------------------------------------

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Observed PM2.5
sc1 = axes[0, 0].scatter(
    spatial_results["longitude"],
    spatial_results["latitude"],
    c=spatial_results["Observed_PM25"],
    s=8,
    alpha=0.8
)
axes[0, 0].set_title("Observed PM2.5 Distribution")
axes[0, 0].set_xlabel("Longitude")
axes[0, 0].set_ylabel("Latitude")
plt.colorbar(sc1, ax=axes[0, 0], label="Observed PM2.5")

# Random Forest predicted PM2.5
sc2 = axes[0, 1].scatter(
    spatial_results["longitude"],
    spatial_results["latitude"],
    c=spatial_results["RF_Predicted_PM25"],
    s=8,
    alpha=0.8
)
axes[0, 1].set_title("Predicted PM2.5 Distribution: Random Forest")
axes[0, 1].set_xlabel("Longitude")
axes[0, 1].set_ylabel("Latitude")
plt.colorbar(sc2, ax=axes[0, 1], label="RF Predicted PM2.5")

# Gradient Boosting predicted PM2.5
sc3 = axes[1, 0].scatter(
    spatial_results["longitude"],
    spatial_results["latitude"],
    c=spatial_results["GB_Predicted_PM25"],
    s=8,
    alpha=0.8
)
axes[1, 0].set_title("Predicted PM2.5 Distribution: Gradient Boosting")
axes[1, 0].set_xlabel("Longitude")
axes[1, 0].set_ylabel("Latitude")
plt.colorbar(sc3, ax=axes[1, 0], label="GB Predicted PM2.5")

# Observed hotspots
hotspots = spatial_results[spatial_results["Observed_Hotspot"] == True]

axes[1, 1].scatter(
    spatial_results["longitude"],
    spatial_results["latitude"],
    s=5,
    alpha=0.25,
    label="Non-hotspot locations"
)

axes[1, 1].scatter(
    hotspots["longitude"],
    hotspots["latitude"],
    s=18,
    alpha=0.9,
    label="Observed PM2.5 hotspots"
)

axes[1, 1].set_title("Observed PM2.5 Hotspots: Top 10% Concentrations")
axes[1, 1].set_xlabel("Longitude")
axes[1, 1].set_ylabel("Latitude")
axes[1, 1].legend()

plt.tight_layout()
plt.savefig("spatial_PM25_distribution_hotspot_collage.png", dpi=300)
plt.show()

In [ ]:
# ------------------------------------------------------------
# 10. Spatial residual collage
# ------------------------------------------------------------

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# RF residual map
sc4 = axes[0].scatter(
    spatial_results["longitude"],
    spatial_results["latitude"],
    c=spatial_results["RF_Residual"],
    s=8,
    alpha=0.8
)

axes[0].set_title("Random Forest Spatial Residuals")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
plt.colorbar(sc4, ax=axes[0], label="Observed - Predicted")

# GB residual map
sc5 = axes[1].scatter(
    spatial_results["longitude"],
    spatial_results["latitude"],
    c=spatial_results["GB_Residual"],
    s=8,
    alpha=0.8
)

axes[1].set_title("Gradient Boosting Spatial Residuals")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
plt.colorbar(sc5, ax=axes[1], label="Observed - Predicted")

plt.tight_layout()
plt.savefig("spatial_residuals_collage_PM25.png", dpi=300)
plt.show()

In [ ]:
# ============================================================
# Interactive Folium Hotspot Maps
# ============================================================

!pip install folium -q

import folium
from folium.plugins import HeatMap

In [ ]:
# ------------------------------------------------------------
# 11. Create observed PM2.5 heatmap
# ------------------------------------------------------------

map_center = [
    spatial_results["latitude"].mean(),
    spatial_results["longitude"].mean()
]

observed_map = folium.Map(
    location=map_center,
    zoom_start=12,
    tiles="CartoDB positron"
)

observed_heat_data = spatial_results[[
    "latitude",
    "longitude",
    "Observed_PM25"
]].dropna().values.tolist()

HeatMap(
    observed_heat_data,
    radius=12,
    blur=18,
    max_zoom=13
).add_to(observed_map)

observed_map.save("observed_PM25_heatmap_Dublin.html")

print("Observed PM2.5 interactive heatmap saved as observed_PM25_heatmap_Dublin.html")

In [ ]:
# ------------------------------------------------------------
# 12. Create Random Forest predicted PM2.5 heatmap
# ------------------------------------------------------------

rf_map = folium.Map(
    location=map_center,
    zoom_start=12,
    tiles="CartoDB positron"
)

rf_heat_data = spatial_results[[
    "latitude",
    "longitude",
    "RF_Predicted_PM25"
]].dropna().values.tolist()

HeatMap(
    rf_heat_data,
    radius=12,
    blur=18,
    max_zoom=13
).add_to(rf_map)

rf_map.save("random_forest_predicted_PM25_heatmap_Dublin.html")

print("Random Forest predicted PM2.5 heatmap saved as random_forest_predicted_PM25_heatmap_Dublin.html")

In [ ]:
# ------------------------------------------------------------
# 13. Create Gradient Boosting predicted PM2.5 heatmap
# ------------------------------------------------------------

gb_map = folium.Map(
    location=map_center,
    zoom_start=12,
    tiles="CartoDB positron"
)

gb_heat_data = spatial_results[[
    "latitude",
    "longitude",
    "GB_Predicted_PM25"
]].dropna().values.tolist()

HeatMap(
    gb_heat_data,
    radius=12,
    blur=18,
    max_zoom=13
).add_to(gb_map)

gb_map.save("gradient_boosting_predicted_PM25_heatmap_Dublin.html")

print("Gradient Boosting predicted PM2.5 heatmap saved as gradient_boosting_predicted_PM25_heatmap_Dublin.html")

In [ ]:
# ------------------------------------------------------------
# 14. Create observed hotspot point map
# ------------------------------------------------------------

hotspot_map = folium.Map(
    location=map_center,
    zoom_start=12,
    tiles="CartoDB positron"
)

for _, row in hotspots.iterrows():
    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=4,
        popup=f"Observed PM2.5: {round(row['Observed_PM25'], 2)}",
        fill=True
    ).add_to(hotspot_map)

hotspot_map.save("observed_PM25_hotspot_points_Dublin.html")

print("Observed PM2.5 hotspot point map saved as observed_PM25_hotspot_points_Dublin.html")


In [ ]:
# ============================================================
# 15. Download all RO3 and RO4 outputs
# ============================================================

from google.colab import files

files.download("random_forest_feature_importance_PM25.csv")
files.download("gradient_boosting_feature_importance_PM25.csv")
files.download("feature_importance_collage_PM25.png")

files.download("spatial_observed_predicted_PM25.csv")
files.download("spatial_PM25_distribution_hotspot_collage.png")
files.download("spatial_residuals_collage_PM25.png")

files.download("observed_PM25_heatmap_Dublin.html")
files.download("random_forest_predicted_PM25_heatmap_Dublin.html")
files.download("gradient_boosting_predicted_PM25_heatmap_Dublin.html")
files.download("observed_PM25_hotspot_points_Dublin.html")